In [15]:
import pandas as pd
import re
from collections import defaultdict, Counter
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pickle
import os
def get_indiv_concepts(formula) -> list:
    concepts = []
    concps = re.findall(r'(?<!\bNOT\s)(?:\b(?:hyp|pre|oth):[^\s)]+)', formula)
    for c in concps:
        try:
            end_idx = c.index(')')
        except:
            end_idx = len(c)
        concepts.append(c[:end_idx])
    return concepts


with open("/workspace/CCE_NLI/code/Abstractions/final_abstractions.pkl", 'rb') as f:
    abs_map = pickle.load(f)
    
def load_csv_data(filepath):
    """Load CSV and extract unit-concept mappings."""
    df = pd.read_csv(filepath)
    unit_concepts = defaultdict(set)
    raw_concepts= []
    for _, row in df.iterrows():
        unit = row['unit']
        formula = row['best_name']
        concepts = get_indiv_concepts(formula)
        
        unit_concepts[unit].update(concepts)
        raw_concepts.extend(concepts)
        
    return unit_concepts, raw_concepts
#how concepts removal same across algortihm


def find_cluster(raw_concept):
    for cluster in abs_map:
        if raw_concept in abs_map[cluster]:
            return cluster
def find_abstractions(expls):
    if isinstance(expls, set):
        glob = expls
    else:
        glob = set([i for j in expls.values() for i in j ])
    abstracts=[]
    exact_concepts_perabs=defaultdict(set)
    for concept in glob:
        raw_concept = concept.split(":")[-1]
        abstraction_cluster = find_cluster(raw_concept)
        if abstraction_cluster==143: 
            continue
        if not abstraction_cluster:abstraction_cluster=150
        exact_concepts_perabs[abstraction_cluster].add(raw_concept)
        
        abstracts.append(abstraction_cluster)
    return Counter(abstracts), set(abstracts), exact_concepts_perabs


In [48]:
import pandas as pd
import os
import re
from pathlib import Path
from collections import defaultdict
import json

class IterativeLotteryAnalyzer:
    def __init__(self, base_path, pretrained_paths=None):
        self.base_path        = Path(base_path)
        self.prune_data       = {}
        self.pretrained_paths = pretrained_paths or {}
        self.pretrained       = {}
        self.dense            = {}
        self.ever_seen        = {}

    def get_indiv_concepts(self, formula) -> list:
        concepts = []
        concps = re.findall(r'(?<!\bNOT\s)(?:\b(?:hyp|pre|oth):[^\s)]+)', formula)
        for c in concps:
            try:
                end_idx = c.index(')')
            except Exception:
                end_idx = len(c)
            concepts.append(c[:end_idx])
        return concepts

    def load_csv_data(self, filepath):
        if not Path(filepath).exists():
            return None
        try:
            df = pd.read_csv(filepath)
            unit_concepts = defaultdict(set)
            for _, row in df.iterrows():
                unit_concepts[row['unit']].update(self.get_indiv_concepts(row['best_name']))
            return unit_concepts
        except Exception as e:
            print(f"  Warning: Could not load {filepath}: {e}")
            return None

    def load_all_csvs(self):
        print("Loading CSV files...")
        for prune_dir in sorted(self.base_path.iterdir()):
            if not prune_dir.is_dir():
                continue
            try:
                prune_level = str(prune_dir.name).split("%")[0]
                float(prune_level)
            except Exception:
                continue

            self.prune_data[prune_level] = {}
            for csv_file in prune_dir.glob('Cluster*'):
                cluster_match = re.search(r'Cluster(\d+)', csv_file.name)
                if not cluster_match:
                    continue
                cluster = cluster_match.group(1)
                try:
                    df = pd.read_csv(csv_file).dropna(subset=['unit', 'best_name'])
                    concepts_per_unit = [set(self.get_indiv_concepts(r)) for r in df['best_name']]
                    unit_to_concept   = dict(zip(df['unit'], concepts_per_unit))
                    all_concepts      = set().union(*concepts_per_unit) if concepts_per_unit else set()
                    self.prune_data[prune_level][cluster] = {
                        'concepts':        all_concepts,
                        'unit_to_concept': unit_to_concept,
                        'total_units':     len(unit_to_concept),
                        'df':              df,
                    }
                except Exception as e:
                    print(f"Error loading {csv_file}: {e}")

        if '0.0' in self.prune_data:
            for cl, d in self.prune_data['0.0'].items():
                self.dense[cl]     = set(d['concepts'])
                self.ever_seen[cl] = set(d['concepts'])

        for cl, path in self.pretrained_paths.items():
            uc = self.load_csv_data(path)
            self.pretrained[cl] = set().union(*uc.values()) if uc else set()

        print(f"Loaded {len(self.prune_data)} pruning levels")
        return self

    # ── analysis ──────────────────────────────────────────────────────────────

    def analyze_iterative_changes(self):
        prune_levels = sorted(self.prune_data.keys(), key=float)
        if len(prune_levels) < 2:
            print("Need at least 2 pruning levels.")
            return {}

        # rows keyed by (sparsity_label, cluster_or_global)
        all_rows = []
        prev_lost_cl     = {}
        prev_global_lost = set()
        baselines        = {}

        for i in range(1, len(prune_levels)):
            prev_level = prune_levels[i-1]
            curr_level = prune_levels[i]
            next_level = prune_levels[i + 1] if i + 1 < len(prune_levels) else None

            prev_data = self.prune_data[prev_level]
            curr_data = self.prune_data[curr_level]
            next_data = self.prune_data.get(next_level, {}) if next_level else {}

            transition = f"{prev_level}→{curr_level}"

            g_prev, g_curr, g_next = set(), set(), set()
            g_lost, g_pres, g_new  = set(), set(), set()

            for cluster in sorted(curr_data.keys(), key=int):
                if cluster not in prev_data:
                    continue

                prev_c = prev_data[cluster]['concepts']
                curr_c = curr_data[cluster]['concepts']
                next_c = next_data.get(cluster, {}).get('concepts', set())

                lost      = prev_c - curr_c
                preserved = prev_c & curr_c
                new_c     = curr_c - prev_c
                relearned = (prev_lost_cl.get(cluster, set()) & curr_c)

                if prev_level == '0.0':
                    baselines[cluster] = set(prev_c)

                n_prev = len(prev_c)
                def pct(s, d=n_prev): return round(100 * len(s) / d, 1) if d else 0.0

                D  = self.dense.get(cluster, set())
                ES = self.ever_seen.get(cluster, set())

                all_rows.append({
                    'transition':            transition,
                    'sparsity':              curr_level,
                    'cluster':               f'Cluster {cluster}',
                    '% lost':                pct(lost),
                    '% relearned':           pct(relearned),          # lost at i-1, back at i
                    '% preserved':           pct(preserved),
                    '% new':                 pct(new_c),
                    # preserved breakdown
                    '% pres: dense&PT':      pct(preserved & D & prev_c,  len(preserved)),
                    '% pres: dense only':    pct(preserved & D - prev_c,  len(preserved)),
                    '% pres: prev_iter only':pct(preserved & prev_c - D, len(preserved)),
                    # new breakdown
                    '% new: truly new':      pct(new_c - D - prev_c - ES, len(new_c)),
                    '% new: dense&prev_c':       pct(new_c & D & prev_c,      len(new_c)),
                    '% new: dense only':     pct(new_c & D - prev_c,      len(new_c)),
                    '% new: prev_c only':        pct(new_c & prev_c - D,      len(new_c)),
                    '% new: seen before':    pct(new_c & ES - prev_c, len(new_c)),
                    # lost breakdown
                    '% lost: dense|prev_c':      pct(lost & (D | prev_c),       len(lost)),
                    '% lost: dense&prev_c':      pct(lost & D & prev_c,       len(lost)),
                    '% lost: dense only':    pct(lost & D - prev_c,       len(lost)),
                    '% lost: prev_c only':       pct(lost & prev_c - D,       len(lost)),
                    '% lost: truly novel':   pct(lost - D - prev_c - ES,  len(lost)),
                    'n_prev': n_prev,
                    'n_curr': len(curr_c),
                })

                self.ever_seen[cluster] = ES | curr_c
                prev_lost_cl[cluster]   = lost

                g_prev |= prev_c;  g_curr |= curr_c;  g_next |= next_c
                g_lost |= lost;    g_pres |= preserved; g_new |= new_c

            # global row
            D_all  = set().union(*self.dense.values())      if self.dense      else set()
            ES_all = set().union(*self.ever_seen.values())  if self.ever_seen  else set()
            g_relearned = prev_global_lost & g_curr

            n_gp = len(g_prev)
            def gpct(s, d=n_gp): return round(100 * len(s) / d, 1) if d else 0.0

            all_rows.append({
                'transition':            transition,
                'sparsity':              curr_level,
                'cluster':               'GLOBAL',
                '% lost':                gpct(g_lost),
                '% relearned':           gpct(g_relearned),
                '% preserved':           gpct(g_pres),
                '% new':                 gpct(g_new),
                '% pres: dense&prev':      gpct(g_pres & D_all & g_prev,            len(g_pres)),
                '% pres: dense only':    gpct(g_pres & D_all - g_prev,            len(g_pres)),
                '% pres: prev_iter only':gpct(g_pres & g_prev - D_all,   len(g_pres)),
                '% new: truly new':      gpct(g_new - D_all - g_prev - ES_all,    len(g_new)),
                '% new: seen before':    gpct(g_new & ES_all - g_prev,            len(g_new)),
                'n_prev': n_gp,
                'n_curr': len(g_curr),
            })

            prev_global_lost = g_lost

        df = pd.DataFrame(all_rows).set_index(['sparsity', 'cluster'])
        return df

    def print_tables(self, df):
        """Print a clean summary table and per-cluster breakdown tables."""
        # ── summary table ─────────────────────────────────────────────────────
        summary_cols = ['% lost', '% relearned', '% preserved', '% new']
        print("\n" + "="*70)
        print("SUMMARY TABLE  (% of concepts at previous level)")
        print("="*70)
        print(df[summary_cols].to_string())

        # ── preserved breakdown ───────────────────────────────────────────────
        pres_cols = [c for c in df.columns if c.startswith('% pres:')]
        print("\n" + "="*70)
        print("PRESERVED BREAKDOWN  (% of preserved concepts)")
        print("="*70)
        print(df[pres_cols].to_string())

        # ── new breakdown ─────────────────────────────────────────────────────
        new_cols = [c for c in df.columns if c.startswith('% new:')]
        print("\n" + "="*70)
        print("NEW CONCEPT BREAKDOWN  (% of new concepts)")
        print("="*70)
        print(df[new_cols].to_string())

        # ── lost breakdown ────────────────────────────────────────────────────
        lost_cols = [c for c in df.columns if c.startswith('% lost:')]
        print("\n" + "="*70)
        print("LOST CONCEPT BREAKDOWN  (% of lost concepts)")
        print("="*70)
        print(df[lost_cols].to_string())

    def concept_survival_rate(self):
        prune_levels = sorted(self.prune_data.keys(), key=float)
        print(f"\n{'='*80}\nCONCEPT SURVIVAL ANALYSIS\n{'='*80}\n")
        baseline_data = self.prune_data[prune_levels[0]]
        all_concepts  = set().union(*[d['concepts'] for d in baseline_data.values()])
        concept_survival = {}
        for concept in all_concepts:
            last_seen = None
            for level in prune_levels:
                if any(concept in d['concepts'] for d in self.prune_data[level].values()):
                    last_seen = level
                elif last_seen is not None:
                    break
            concept_survival[concept] = last_seen or prune_levels[0]
        survival_groups = defaultdict(list)
        for concept, last_level in concept_survival.items():
            survival_groups[last_level].append(concept)
        for i, level in enumerate(prune_levels[1:], 1):
            lost = survival_groups.get(prune_levels[i - 1], [])
            if lost:
                print(f"  Lost between {prune_levels[i-1]}% → {level}%: {len(lost)} concepts")
                for c in sorted(lost)[:10]: print(f"    - {c}")
                if len(lost) > 10: print(f"    ... and {len(lost)-10} more")
        return survival_groups

    def cluster_transfer_analysis(self):
        prune_levels = sorted(self.prune_data.keys(), key=float)
        print(f"\n{'='*80}\nCLUSTER TRANSFER ANALYSIS\n{'='*80}\n")
        for i in range(len(prune_levels) - 1):
            prev_level, curr_level = prune_levels[i], prune_levels[i + 1]
            print(f"\nFrom {prev_level}% → {curr_level}%:")
            prev_cc, curr_cc = defaultdict(set), defaultdict(set)
            for cl, d in self.prune_data[prev_level].items():
                for c in d['concepts']: prev_cc[c].add(cl)
            for cl, d in self.prune_data[curr_level].items():
                for c in d['concepts']: curr_cc[c].add(cl)
            transferred = [{'concept': c, 'from': sorted(prev_cc[c]), 'to': sorted(curr_cc[c])}
                           for c in curr_cc if c in prev_cc and prev_cc[c] != curr_cc[c]]
            if transferred:
                print(f"  Concepts that moved clusters: {len(transferred)}")
                for item in transferred[:15]:
                    print(f"    '{item['concept']}': {item['from']} → {item['to']}")
                if len(transferred) > 15:
                    print(f"    ... and {len(transferred)-15} more")
            else:
                print("  No concepts transferred between clusters")

    def save_results(self, df, output_file='iterative_analysis_results.csv'):
        df.to_csv(output_file)
        print(f"\nResults saved to {output_file}")
    def lost_concept_redundancy(self):
        """
        For each pruning transition, record every concept that was lost and
        how redundant it was in the previous iteration.

        Redundancy of a concept at level i = number of units that expressed it
        (i.e. how many units had that concept in their unit_to_concept mapping).

        Returns
        -------
        dict:  {transition_str: {cluster: [ {concept, redundancy, units_expressing} ] }}
        """
        prune_levels = sorted(self.prune_data.keys(), key=float)
        if len(prune_levels) < 2:
            print("Need at least 2 pruning levels.")
            return {}

        results = {}

        for i in range(1, len(prune_levels)):
            prev_level = prune_levels[0]
            curr_level = prune_levels[i]
            transition = f"{prev_level}→{curr_level}"
            results[transition] = {}

            prev_data = self.prune_data[prev_level]
            curr_data = self.prune_data[curr_level]

            print(f"\n{'='*70}")
            print(f"  Lost concept redundancy:  {transition}")
            print(f"{'='*70}")

            for cluster in sorted(curr_data.keys(), key=int):
                if cluster not in prev_data:
                    continue

                prev_cluster  = prev_data[cluster]
                curr_concepts = curr_data[cluster]['concepts']
                prev_concepts = prev_cluster['concepts']
                u2c           = prev_cluster['unit_to_concept']   # unit -> set(concepts)

                lost=prev_concepts - curr_concepts
                _, lostabs, _ = find_abstractions(prev_concepts - curr_concepts)
                print(lostabs)
                concept_records = []
                
                for a in lostabs:
                    units_expressing=[]
                    actu_lost=[]
                    for concept in abs_map[a]:
                        units_expressing.extend([
                            unit for unit, concepts in u2c.items()
                            if f'pre:tok:{concept}' in concepts or  f'hyp:tok:{concept}' in concepts 
                        ])
                        if concept in lost: actu_lost.append(concept)
                        
                    red = len(set(units_expressing))
                    if red==0: continue
                    concept_records.append({
                        'concept':           a,
                        'redundancy':        red,      # n units that had it
                        'actu_lost':  actu_lost,
                    })
                    
                # for each lost concept, count how many units expressed it
                
                '''for concept in sorted(lost):
                    units_expressing = [
                        unit for unit, concepts in u2c.items()
                        if concept in concepts
                    ]
                    redundancy = len(units_expressing)
                    concept_records.append({
                        'concept':           concept,
                        'redundancy':        redundancy,      # n units that had it
                        'units_expressing':  units_expressing,
                    })'''

                # sort by redundancy descending so highly redundant lost concepts come first
                concept_records.sort(key=lambda x: x['redundancy'], reverse=True)
                results[transition][cluster] = concept_records

                # ── print summary ─────────────────────────────────────────────
                if concept_records:
                    red_vals = [r['redundancy'] for r in concept_records]
                    print(f"\n  Cluster {cluster}:  {len(lost)} lost concepts  |  "
                          f"avg redundancy={np.mean(red_vals):.1f}  "
                          f"max={max(red_vals)}  min={min(red_vals)}")
                    print(f"  {'concept':<40}  {'n_units':>7}")
                    print(f"  {'-'*50}")
                    for r in concept_records:
                        print(f"  {r['concept']:<40}  {r['redundancy']:>7}")
                else:
                    print(f"\n  Cluster {cluster}: no lost concepts")

            # ── global summary across clusters ────────────────────────────────
            all_lost_records = [r for cl_recs in results[transition].values() for r in cl_recs]
            if all_lost_records:
                red_vals = [r['redundancy'] for r in all_lost_records]
                print(f"\n  GLOBAL: {len(all_lost_records)} lost concepts  |  "
                      f"avg redundancy={np.mean(red_vals):.1f}  "
                      f"max={max(red_vals)}  min={min(red_vals)}")

        return results


    def lost_redundancy_df(self):
        """
        Same as lost_concept_redundancy but returns a flat DataFrame:
        columns: transition, cluster, concept, redundancy
        """
        raw = self.lost_concept_redundancy()
        rows = []
        for transition, clusters in raw.items():
            for cluster, records in clusters.items():
                for r in records:
                    rows.append({
                        'transition': transition,
                        'cluster':    f'Cluster {cluster}',
                        'concept':    r['concept'],
                        'redundancy': r['redundancy'],
                    })
        df = pd.DataFrame(rows)
        return df

In [50]:
if __name__ == "__main__":
    analyzer = IterativeLotteryAnalyzer(
        base_path="/workspace/CCE_NLI/BERT/exp/CoFi/Run0.25_5/Expls",
        
    )
    analyzer.load_all_csvs()

    '''df = analyzer.analyze_iterative_changes()
    analyzer.print_tables(df)
    analyzer.save_results(df)
    analyzer.concept_survival_rate()
    analyzer.cluster_transfer_analysis()'''
    #analyzer.load_all_csvs()

    # detailed dict with unit lists
    raw = analyzer.lost_concept_redundancy()

    # flat DataFrame for easy filtering/plotting
    df = analyzer.lost_redundancy_df()

    # e.g. concepts lost at 25% with redundancy > 3
    df[(df['transition'] == '0.0→25.0') & (df['redundancy'] > 3)]

Loading CSV files...
Loaded 6 pruning levels

  Lost concept redundancy:  0.0→0.25267370011269075
{129, 58, 6, 135, 122, 73, 102, 38, 12, 10, 79, 82, 150, 88, 24, 56, 31}

  Cluster 1:  23 lost concepts  |  avg redundancy=7.7  max=35  min=1
  concept                                   n_units
  --------------------------------------------------
  6                                              35
  73                                             22
  102                                            12
  56                                             11
  135                                             9
  129                                             8
  12                                              5
  122                                             3
  82                                              3
  79                                              2
  58                                              1
  38                                              1
  10                          